In [2]:
import jax

jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
from copy import deepcopy
from scipy.signal import welch
from jimgw.core.single_event.data import Data, PowerSpectrum
from jimgw.core.single_event.detector import get_H1, get_detector_preset
from jimgw.core.single_event.waveform import RippleIMRPhenomD, RippleIMRPhenomPv2
from jimgw.core.single_event.gps_times import (
    greenwich_mean_sidereal_time as compute_gmst,
)
from jimgw.core.single_event.transforms import (
    MassRatioToSymmetricMassRatioTransform,
    SphereSpinToCartesianSpinTransform
)

In [7]:
gps_time = 1126259462.0
gmst = compute_gmst(gps_time)

injection_parameters = {
    "M_c": 31.0,
    "q": 0.87,
    "s1_mag": 0.25,
    "s1_theta": 1.65,
    "s1_phi": 3.13,
    "s2_mag": 0.29,
    "s2_theta": 1.78,
    "s2_phi": 3.14,
    "ra": 2.02,
    "dec": -1.24,
    "psi": 1.59,
    "d_L": 8000.0,
    "iota": 2.71,
    "phase_c": 3.11,
    "t_c": 0.02,
    "trigger_time": gps_time,
    "gmst": gmst,
}

_inj_params = injection_parameters.copy()
_inj_params = MassRatioToSymmetricMassRatioTransform.forward(_inj_params)
_inj_params = SphereSpinToCartesianSpinTransform("s1").forward(_inj_params)
_inj_params = SphereSpinToCartesianSpinTransform("s2").forward(_inj_params)
injection_parameters.update(_inj_params)

f_min = 20.0
f_max = 1024.0
duration = 16.0
# duration = 10.0
sampling_frequency = f_max * 2

# initialize waveform
PhenomPv2 = RippleIMRPhenomPv2(f_ref=20)

ifo_list = get_detector_preset()
ifos = ifo_list['H1'], ifo_list['L1']
for ifo in ifos:
    # psd_file = 'ET_D_psd.txt'
    ifo.load_and_set_psd()
    ifo.frequency_bounds = (f_min, f_max)
    # ifo.set_frequency_bounds(f_min, f_max)
    ifo.inject_signal(
        duration,
        sampling_frequency,
        0.0,
        PhenomPv2,
        injection_parameters,
        is_zero_noise=False,
    )

Grabbing GWTC-2 PSD for H1
For detector H1, the injected signal has:
  - Optimal SNR: 2.0065
  - Match filtered SNR: 2.7317+0.4684j
Grabbing GWTC-2 PSD for L1
For detector L1, the injected signal has:
  - Optimal SNR: 1.8762
  - Match filtered SNR: 2.4083+0.7150j
